# check_paralle.ipynb

Run this **after the first VM has started `training.py`** to confirm the switch to the
coordinated multi-VM mode worked. It checks four things:

1. **O_EXCL works on this MooseFS mount** — the whole claim protocol rests on an exclusive
   create being atomic. (Single-VM proves the mechanism; true cross-client needs 2 VMs.)
2. **Coordination is active** — this VM registered in `VM_parallel/` with a *fresh lease*.
3. **Migration succeeded** — every pre-existing checkpoint is recognized as done and is
   **not** being re-claimed for a re-run.
4. **Live progress** — how many combos are done / claimed / free, and by which VM.

In [ ]:
# Config -- point at the SHARED coordinated out_dir (the same dir the current run's
# checkpoints already live in). Must match training.ipynb's OUT_DIR.
import sys, os, json, time
from pathlib import Path

REPO = '/workspace/stable-query-latent'
OUT_DIR = os.path.join(REPO, 'VICReg_review/heads/cloud_full_sweep_a100')
SWEEP_YAML = os.path.join(REPO, 'VICReg_review/sweep/sweep.yaml')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from VICReg_review.sweep.config import SweepConfig
from VICReg_review.sweep import jobspec
from VICReg_review.sweep import coordination as coord

ROOT = Path(OUT_DIR)
print('repo    :', REPO)
print('out_dir :', OUT_DIR, '(exists:', ROOT.exists(), ')')

In [ ]:
# 1. O_EXCL atomicity self-test on THIS filesystem (the claim primitive).
#    Create a file exclusively twice -> the second MUST fail. If it doesn't, the
#    network FS is not honouring exclusive create and coordination is unsafe here.
probe = ROOT / '_coord_probe'
probe.mkdir(parents=True, exist_ok=True)
tf = probe / f'oexcl_{int(time.time()*1000)}.tmp'
first = coord._atomic_create(tf, {'ok': 1})
second = coord._atomic_create(tf, {'ok': 2})
try:
    tf.unlink()
except OSError:
    pass
print('O_EXCL create: first =', first, ' second =', second)
print('VERDICT:', 'OK (exclusive create is atomic here)' if (first and not second)
      else 'FAIL -- exclusive create is NOT reliable on this mount; coordination unsafe')

In [ ]:
# 2. Coordination active? List registered VMs + whether their lease is fresh.
vm_dir = ROOT / coord.VM_DIR
vms = sorted(vm_dir.glob('*.json')) if vm_dir.exists() else []
if not vms:
    print('NO VM_parallel/ entries found ->',
          'coordination is NOT active. Either training.py is not the coordinated build,',
          'or no VM has started yet.')
else:
    now = time.time()
    print(f'{len(vms)} VM(s) registered in {vm_dir}:')
    for f in vms:
        rec = coord._read_json(f) or {}
        exp = rec.get('expiry')
        if exp is not None:
            fresh = f'lease {exp - now:+.0f}s' + (' (FRESH)' if exp > now else ' (EXPIRED)')
        else:
            age = now - f.stat().st_mtime
            fresh = f'mtime {age:.0f}s ago'
        print(f"  - {rec.get('vm', f.stem):20} pid={rec.get('pid')} {fresh}  info={rec.get('info', {})}")

In [ ]:
# 3 + 4. Migration + live progress. For every combo in the grid: does it have a
#        checkpoint (old run -> should be treated as done), a done.json (new marker),
#        a status.json (claimed now, by whom)?
cfg = SweepConfig.load(SWEEP_YAML)
cfg.out_dir = OUT_DIR
combos = list(cfg.iter_combos())

n_ckpt = n_marker = n_claimed = 0
claimants = {}
migration_bad = []   # checkpoint combos that are being re-claimed (should never happen)
for c in combos:
    cdir = ROOT / c.combo_id
    has_ckpt = jobspec.combo_paths(cfg, c)['checkpoint'].exists()
    has_marker = (cdir / coord.DONE).exists()
    st = coord._read_json(cdir / coord.STATUS)
    if has_ckpt:
        n_ckpt += 1
    if has_marker:
        n_marker += 1
    if st and not has_marker and not has_ckpt:
        n_claimed += 1
        claimants[st.get('vm')] = claimants.get(st.get('vm'), 0) + 1
    if has_ckpt and st and not has_marker:
        migration_bad.append(c.combo_id)   # done work being re-claimed = migration FAILURE

total = len(combos)
done = n_ckpt + n_marker
print(f'grid           : {total} combos')
print(f'done (old ckpt): {n_ckpt}   <- recognised from the pre-migration run')
print(f'done (new mark): {n_marker}')
print(f'claimed now    : {n_claimed}  by {claimants or {}}')
print(f'free / remaining: {total - done - n_claimed}')
print()
if migration_bad:
    print('MIGRATION FAIL: these already-checkpointed combos are being re-claimed (out_dir wrong?):')
    print('  ', migration_bad[:12], '...' if len(migration_bad) > 12 else '')
else:
    print('MIGRATION OK: every pre-existing checkpoint is recognised as done; none re-claimed.')